In [0]:
%run ./schemas

In [0]:

# SCHEMA DRIFT DIAGNOSTIC
# Uses synthetic JSONL and your actual Raw schema.
# Does NOT write to production Bronze/Silver tables.

import json
import uuid
from pathlib import Path
from pyspark.sql import functions as F

qa_dir = Path(
    "/Volumes/github_health/raw/bronze_data/_qa_tests/drift_probe"
)
qa_dir.mkdir(parents=True, exist_ok=True)

qa_file = qa_dir / f"issues_{uuid.uuid4().hex[:8]}.jsonl"

base = {
    "number": 1,
    "title": "Synthetic schema drift test",
    "state": "open",
    "created_at": "2026-10-08T10:00:00Z",
    "updated_at": "2026-10-08T11:00:00Z",
    "_repo": "apache/spark"
}

records = [
    # 1. Completely valid record
    {**base, "id": 990000000001},

    # 2. New unexpected column
    {
        **base,
        "id": 990000000002,
        "surprise_new_field": "unexpected"
    },

    # 3. Wrong data type (id should be numeric)
    {**base, "id": "not_an_integer"}
]

qa_file.write_text(
    "\n".join(json.dumps(r) for r in records) + "\n",
    encoding="utf-8"
)


# SERVERLESS-COMPATIBLE SCHEMA DRIFT TEST
# No caching and no writes to production tables

from pyspark.sql import functions as F

# Read JSON lines as text, then parse using your actual schema
raw_lines = spark.read.text(str(qa_file))

raw = (
    raw_lines
    .select(
        F.col("value").alias("original_json"),
        F.from_json(
            F.col("value"),
            RAW_SCHEMAS["issues"],
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_corrupt_record"
            }
        ).alias("parsed")
    )
    .select("original_json", "parsed.*")
)

print("========== RESULTS ==========")

display(raw.select("id", "title", "_corrupt_record"))

good = raw.filter(
    F.col("_corrupt_record").isNull()
).count()

bad = raw.filter(
    F.col("_corrupt_record").isNotNull()
).count()

extra_ignored = (
    "surprise_new_field" not in raw.columns
    and raw.filter(
        (F.col("id") == 990000000002)
        & F.col("_corrupt_record").isNull()
    ).count() == 1
)

print("Total input:", raw.count())
print("Accepted by parser:", good)
print("Detected as corrupt:", bad)
print("Unexpected field silently ignored:", extra_ignored)

if extra_ignored:
    print("GAP: New columns are not being detected.")
else:
    print("Review result: New column was not silently ignored.")


In [0]:

# CELL 3: INSPECT REAL GITHUB RAW FIELDS
# Read-only: does not change production data

import json
from pathlib import Path
from collections import defaultdict

batch = "2524cfd1-59e1-4e7f-8a9c-90ee49f49d89"

manifest_dir = Path(
    "/Volumes/github_health/raw/bronze_data/bronze/_manifests"
)

matches = list(
    manifest_dir.glob(f"full_load_*_{batch}.json")
)

assert len(matches) == 1, "Historical manifest not found uniquely"

manifest = json.loads(matches[0].read_text())

extra_fields = defaultdict(set)
sample_counts = defaultdict(int)

for item in manifest["files"]:
    entity = item["entity"]

    if entity not in RAW_SCHEMAS:
        continue

    declared = set(RAW_SCHEMAS[entity].fieldNames())

    with Path(item["path"]).open(encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 20:
                break

            if not line.strip():
                continue

            record = json.loads(line)

            if not isinstance(record, dict):
                continue

            extra_fields[entity].update(
                set(record.keys()) - declared
            )

            sample_counts[entity] += 1

print("========== REAL RAW SCHEMA INSPECTION ==========")

for entity in sorted(sample_counts):
    print(f"\nEntity: {entity}")
    print(f"Sampled records: {sample_counts[entity]}")
    print(f"Undeclared top-level fields: {sorted(extra_fields[entity])}")



In [0]:

# CELL 4: DISCOVER EXISTING SOURCE FIELDS
# Read-only; does not change Bronze or Silver

import json
from pathlib import Path
from collections import defaultdict

manifest_dir = Path(
    "/Volumes/github_health/raw/bronze_data/bronze/_manifests"
)

full_batch = "2524cfd1-59e1-4e7f-8a9c-90ee49f49d89"

manifest_paths = (
    list(manifest_dir.glob(f"full_load_*_{full_batch}.json"))
    + list(manifest_dir.glob("incremental_*.json"))
)

assert manifest_paths, "No manifests found"

known_fields = defaultdict(set)
record_counts = defaultdict(int)
file_counts = defaultdict(int)
malformed_counts = defaultdict(int)
seen_files = set()

for manifest_path in manifest_paths:
    manifest = json.loads(manifest_path.read_text())

    for item in manifest["files"]:
        entity = item["entity"]
        file_path = Path(item["path"])

        if entity not in RAW_SCHEMAS:
            continue

        if str(file_path) in seen_files:
            continue

        seen_files.add(str(file_path))
        file_counts[entity] += 1

        with file_path.open(encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    malformed_counts[entity] += 1
                    continue

                if isinstance(record, dict):
                    known_fields[entity].update(record.keys())
                    record_counts[entity] += 1

print("========== COMPLETE SOURCE FIELD SCAN ==========")

for entity in sorted(known_fields):
    declared = set(RAW_SCHEMAS[entity].fieldNames())
    known = known_fields[entity]

    print(f"\n{entity}")
    print("Files inspected:", file_counts[entity])
    print("Records inspected:", record_counts[entity])
    print("Distinct source fields:", len(known))
    print("Fields beyond modeled schema:", len(known - declared))
    print("Malformed JSON lines:", malformed_counts[entity])

print("\nTotal files inspected:", len(seen_files))
print("Manifests inspected:", len(manifest_paths))


In [0]:

# CELL 5: SAVE KNOWN SOURCE FIELD CONTRACT
# Does not modify Bronze, Silver or quarantine tables

import json
from pathlib import Path

contract_path = Path(
    "/Volumes/github_health/raw/bronze_data/bronze/"
    "_contracts/known_source_fields_v1.json"
)

contract = {
    "version": 1,
    "description": "Approved top-level fields observed in historical GitHub data",
    "source_manifests": sorted(
        p.name for p in manifest_paths
    ),
    "allowed_fields": {
        entity: sorted(fields)
        for entity, fields in sorted(known_fields.items())
    }
}

contract_path.parent.mkdir(parents=True, exist_ok=True)

if contract_path.exists():
    print("Contract already exists; not overwriting.")
else:
    contract_path.write_text(
        json.dumps(contract, indent=2),
        encoding="utf-8"
    )
    print("Source schema contract created successfully.")

saved = json.loads(
    contract_path.read_text(encoding="utf-8")
)

print("\n========== CONTRACT VERIFICATION ==========")

for entity, fields in saved["allowed_fields"].items():
    print(f"{entity}: {len(fields)} approved fields")

print("\nContract location:", contract_path)
